# Sevastopol AI — парсер каналов (одна ячейка)

Читает любые **публичные** каналы с твоего личного аккаунта (Telethon) и копирует
новые посты в Избранное (или в указанный чат).

1. Возьми `API_ID` / `API_HASH`: https://my.telegram.org → API development tools.
2. Впиши их в ячейку ниже; `SESSION` оставь пустым — ячейка сама спросит телефон
   и код из Telegram, напечатает строку сессии и запомнит её.
3. Нажми ▶️. Остановка — ■.

⚠️ **Строка сессии = полный доступ к аккаунту.** Не публикуй ноутбук с ней и никому
её не показывай. Скомпрометирована — Telegram → Настройки → Устройства → заверши
сессию и запусти ячейку заново.

In [ ]:
# ═══════════ Sevastopol AI — парсер каналов одной ячейкой ═══════════
API_ID = 0                                     # ← my.telegram.org → API development tools
API_HASH = ""                                  # ← оттуда же
SESSION = ""                                   # пусто → ячейка спросит телефон и код сама

CHANNELS = ["https://t.me/Sevastopol_AI"]      # любые публичные каналы (ссылка или @имя)
TARGET = "me"                                  # "me" = Избранное, либо "@имя" / "-1001234567890"
POLL_SECONDS = 12                              # пауза между кругами опроса
INCLUDE_HISTORY = False                        # True — на старте скопировать последние ~15 постов

import json, os, pathlib, shutil, subprocess, sys

REPO = "https://github.com/Yurich-citycode/SevastopolAIbot.git"
DIR = "/content/SevastopolAIbot"


def sh(*cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


# 1) код + telethon
if os.path.isdir(DIR):
    if subprocess.run(["git", "-C", DIR, "pull", "--ff-only", "origin", "main"]).returncode:
        shutil.rmtree(DIR)                      # локальные правки мешают — клонируем заново
if not os.path.isdir(DIR):
    sh("git", "clone", "--depth", "1", REPO, DIR)
sh(sys.executable, "-m", "pip", "install", "-q", "telethon")

# 2) список каналов и настройки (events/sources.json)
(pathlib.Path(DIR) / "events" / "sources.json").write_text(json.dumps({
    "target": TARGET,
    "channels": CHANNELS,
    "poll_seconds": POLL_SECONDS,
    "include_history": INCLUDE_HISTORY,
}, ensure_ascii=False, indent=2), encoding="utf-8")

os.environ["TELETHON_API_ID"] = str(API_ID)
os.environ["TELETHON_API_HASH"] = API_HASH
os.environ["FORWARDER_TARGET"] = TARGET

sys.path.insert(0, os.path.join(DIR, "events"))
import forwarder as fw

# 3) вход, если строки сессии ещё нет (спросит телефон, код из Telegram, 2FA)
if not SESSION.strip():
    from telethon import TelegramClient
    from telethon.sessions import StringSession

    client = TelegramClient(StringSession(), API_ID, API_HASH)
    client.parse_mode = None
    await client.start()
    me = await client.get_me()
    SESSION = client.session.save()
    print("Аккаунт: %s (@%s, id %s)" % (me.first_name, me.username, me.id))
    print("\nTELETHON_SESSION — сохрани, чтобы не входить заново:")
    print(SESSION)
    await client.disconnect()
os.environ["TELETHON_SESSION"] = SESSION.strip()

# 4) запуск — живёт, пока не нажмёшь ■; прогресс в events/forwarder_state.json
await fw.run_async(reset=False, once=False)